# Modeling — IEEE-CIS Fraud Detection

This notebook loads the **processed** features saved by `ieee_cis_fraud_detection/features.py`
(`data/processed/*.parquet`), sets up a **temporal train/validation split** (the data is
time-ordered by `TransactionDT`), and provides an **MLflow** harness for tracking experiments.

You build the models — the cells below give you:
1. Loaded features + temporal split (`X_train`, `y_train`, `X_val`, `y_val`)
2. An MLflow setup + a `train_and_eval()` helper that logs validation AUC
3. Template examples to verify the harness works end-to-end

In [1]:
import numpy as np
import pandas as pd

import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ieee_cis_fraud_detection.config import PROCESSED_DATA_DIR, PROJ_ROOT

/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-27 00:38:10.917 | INFO     | ieee_cis_fraud_detection.config:<module>:11 - PROJ_ROOT path is: /Users/alex/IEEE-CIS_Fraud_Detection_MLOp


In [2]:
transaction = pd.read_parquet(PROCESSED_DATA_DIR / "train_transaction_filtered.parquet")
identity = pd.read_parquet(PROCESSED_DATA_DIR / "train_identity_filtered.parquet")

print("train_transaction:", transaction.shape)
print("train_identity:  ", identity.shape)
print(
    "Categorical cols (transaction):",
    [c for c in transaction.columns if transaction[c].dtype.name == "category"],
)

train_transaction: (590540, 220)
train_identity:   (144233, 29)
Categorical cols (transaction): ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M6']


## Baseline scope: transaction-only

We start with a **transaction-only** baseline and
left-join `identity` later if performance plateaus. The `identity` table is
loaded above and kept ready for that step.

In [3]:
TARGET = "isFraud"
# Pure ID column — never a feature. TransactionDT is kept as a feature for now
# (you may drop it or engineer hour/weekday features from it).
DROP_COLS = ["TransactionID"]


def prepare_data(df: pd.DataFrame):
    """Split a transaction frame into features and labels."""
    y = df[TARGET].astype(int).to_numpy()
    X = df.drop(columns=[TARGET] + DROP_COLS)
    return X, y


def temporal_split(df: pd.DataFrame, val_frac: float = 0.2):
    """Split by time (TransactionDT), NOT randomly — avoids time leakage."""
    df = df.sort_values("TransactionDT").reset_index(drop=True)
    split_idx = int(len(df) * (1 - val_frac))
    return df.iloc[:split_idx], df.iloc[split_idx:]


train_df, val_df = temporal_split(transaction, val_frac=0.2)
X_train, y_train = prepare_data(train_df)
X_val, y_val = prepare_data(val_df)

print(f"train: {X_train.shape}  val: {X_val.shape}")
print(f"val fraud rate: {y_val.mean():.4f}")

train: (472432, 218)  val: (118108, 218)
val fraud rate: 0.0344


## Why a temporal split?

`TransactionDT` is a timedelta from a fixed reference datetime, so rows are
**time-ordered**. A random KFold would place the same card/device in both train
and validation, inflating AUC with leakage. Sorting by `TransactionDT` and
holding out the last 20% of time gives an honest estimate of future fraud.

> Once defined here, this split is shared by every model family so all AUCs are
> directly comparable.

In [4]:
# --- MLflow setup ---------------------------------------------------------
# MLflow 3.x requires a database backend; sqlite keeps it fully local.
TRACKING_DB = PROJ_ROOT / "mlruns" / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB}")
mlflow.set_experiment("ieee-fraud-detection")
mlflow.autolog()  # auto-logs params/metrics for sklearn / lightgbm / xgboost / catboost


2026/08/27 00:38:13 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [5]:
def train_and_eval(model, X_tr, y_tr, X_va, y_va, run_name):
    """Fit a model inside an MLflow run and log the validation AUC.

    Uses predict_proba when the model has it (LR / LDA / trees); falls back to
    decision_function for models without it (e.g. LinearSVC). AUC is rank-based,
    so raw scores from decision_function are fine.
    """
    with mlflow.start_run(run_name=run_name):
        model.fit(X_tr, y_tr)
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_va)[:, 1]
        else:
            y_score = model.decision_function(X_va)
        auc = roc_auc_score(y_va, y_score=y_score)  # kwarg is y_score in sklearn 1.9.0
        mlflow.log_metric("val_auc", auc)
        print(f"{run_name}: val AUC = {auc:.4f}")
        return model, auc

## Build your own models

Call `train_and_eval(model, X_train, y_train, X_val, y_val, run_name="...")`
for each model you want to compare. Every run is logged to MLflow with its
`val_auc`.

**Two tracks (per our plan):**

- **Tree track (NaN-native)** — LightGBM / XGBoost / CatBoost can be fed
  `X_train` directly; they handle NaN and the `category` dtype natively.
- **Classical track** — LR / LDA / SVM / MLP need the `preprocessor` below
  (impute → one-hot categoricals → scale numerics).

The next cells are templates to verify the harness; replace/extend them freely.

In [6]:
# Shared preprocessing for the classical track (impute -> encode -> scale).
# IMPORTANT: fits only on X_train, so no validation information leaks.
#
# Missingness-aware imputation (MNAR): missing values carry signal here, so we
# never throw that information away:
#   - numerics:     median fill + binary is_missing indicator per column
#                   (add_indicator=True -> the indicator carries the signal)
#   - categoricals: NaN becomes its own one-hot category (constant "missing"
#                   fill -> OneHotEncoder); the category IS the indicator
categorical_features = X_train.select_dtypes(include="category").columns.tolist()
numeric_features = [c for c in X_train.columns if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)
print(
    f"numeric features: {len(numeric_features)} | "
    f"categorical features: {len(categorical_features)}"
)

numeric features: 209 | categorical features: 9


In [18]:
# EXAMPLE (classical track) — verify the harness, then tune/replace.
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])
model_lr, auc_lr = train_and_eval(lr_pipeline, X_train, y_train, X_val, y_val, "baseline_logreg")

2026/08/27 05:36:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 05:38:12 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_

baseline_logreg: val AUC = 0.8181


In [8]:
# BASELINE (classical track) — LDA via the missingness-aware preprocessor.
# Scaling matters for LDA's shared-covariance estimate; the preprocessor handles it.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LinearDiscriminantAnalysis()),
])
model_lda, auc_lda = train_and_eval(lda_pipeline, X_train, y_train, X_val, y_val, "baseline_lda")

2026/08/27 00:42:09 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 00:42:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_

baseline_lda: val AUC = 0.8027


In [9]:
# BASELINE (classical track) — LinearSVC.
# No predict_proba -> train_and_eval falls back to decision_function for AUC.
from sklearn.svm import LinearSVC

svm_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LinearSVC(max_iter=2000)),
])
model_svm, auc_svm = train_and_eval(svm_pipeline, X_train, y_train, X_val, y_val, "baseline_linear_svc")

2026/08/27 00:43:53 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 01:01:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_

baseline_linear_svc: val AUC = 0.8137


In [16]:
# BASELINE — RandomForest (uses the classical preprocessor).
# sklearn's RandomForest can't take NaN or the `category` dtype natively, so it
# goes through the missingness-aware preprocessor (median + indicator, one-hot).
# Scaling is harmless for trees; one-hot of categoricals is slightly suboptimal
# vs label-encoding, but fine for a no-tuning baseline.
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)),
])
model_rf, auc_rf = train_and_eval(rf_pipeline, X_train, y_train, X_val, y_val, "baseline_random_forest")

2026/08/27 05:30:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 05:33:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_

baseline_random_forest: val AUC = 0.8902


In [19]:
# EXAMPLE (tree track) — NaN-native, uses the `category` dtype directly.
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05)
model_lgb, auc_lgb = train_and_eval(lgb_model, X_train, y_train, X_val, y_val, "baseline_lgbm")

2026/08/27 05:39:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.118717 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16051
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 218
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784


2026/08/27 05:39:34 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 05:39:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/27 05:39:

baseline_lgbm: val AUC = 0.9006


In [11]:
# BASELINE (tree track) — XGBoost, NaN-native.
# Needs enable_categorical=True to use the pandas `category` dtype.
import xgboost as xgb

xgb_model = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, enable_categorical=True)
model_xgb, auc_xgb = train_and_eval(xgb_model, X_train, y_train, X_val, y_val, "baseline_xgb")

2026/08/27 05:11:43 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/08/27 05:12:01 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 05:12:01 WARNING m

baseline_xgb: val AUC = 0.8960


In [14]:
# BASELINE (tree track) — CatBoost, NaN-native.
# CatBoost needs: (1) category columns listed via cat_features, and (2) NaN in
# categorical features converted to a string (unlike LightGBM/XGBoost, CatBoost
# rejects NaN there). Numeric NaN is handled natively.
from catboost import CatBoostClassifier

cb_cat_features = X_train.select_dtypes(include="category").columns.tolist()


def cb_prep(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in cb_cat_features:
        df[c] = df[c].astype("object").fillna("missing")
    return df


X_train_cb = cb_prep(X_train)
X_val_cb = cb_prep(X_val)

cb_model = CatBoostClassifier(
    iterations=200, learning_rate=0.05, verbose=0, cat_features=cb_cat_features
)
model_cb, auc_cb = train_and_eval(cb_model, X_train_cb, y_train, X_val_cb, y_val, "baseline_catboost")

baseline_catboost: val AUC = 0.8696


## Compare runs

Run metadata is stored in `mlruns/mlflow.db` (SQLite) and model artifacts under
`mlruns/`. To open the UI, run `mlflow ui` from the project root and open the
printed URL.


In [22]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp = client.get_experiment_by_name("ieee-fraud-detection")
runs = client.search_runs([exp.experiment_id], order_by=["start_time DESC"])

# One row per model: best val_auc among runs that actually logged it.
# Drops leftover runs from failed executions (no val_auc) and collapses
# duplicates created by re-running a model cell.
best = {}
for r in runs:
    name = r.data.tags.get("mlflow.runName", "?")
    auc = r.data.metrics.get("val_auc", float("nan"))
    if auc == auc:  # skip NaN (auc == auc is False only for NaN)
        best[name] = max(best.get(name, float("-inf")), auc)

print(f"{'run':<24} {'best_val_auc':>12}")
print("-" * 38)
for name, auc in sorted(best.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{name:<24} {auc:>12.4f}")
print(f"\n({len(runs)} total runs -> {len(best)} models shown)")

run                      best_val_auc
--------------------------------------
baseline_lgbm                  0.9006
baseline_xgb                   0.8960
baseline_random_forest         0.8902
baseline_catboost              0.8696
baseline_logreg                0.8181
baseline_linear_svc            0.8137
baseline_lda                   0.8027

(7 total runs -> 7 models shown)


The baseline model results shows that tree based model performs the best, therefore, we will further finetune the tree based models. We would try to use LGBM and XGB with the `identity` dataset (as they handles missing values well) and see if the AUC would improve.

## Add `identity` to the feature set

`identity` (device/network data) is present for only ~25% of transactions, so we
left-join it onto `transaction` on `TransactionID` **before** the temporal split
— identity rows are static per transaction (no cross-row aggregates), hence no
leakage.

For the remaining ~75% of rows the identity columns are NaN — itself a strong
MNAR signal for trees ("no identity/device data recorded"). Identity categoricals
(`DeviceType`, `DeviceInfo`) stay as the pandas `category` dtype (NaN native);
`DeviceInfo` is very high-cardinality, so we do **not** one-hot it.

In [23]:
# Left-join identity onto transaction BEFORE the temporal split (no leakage).
transaction_id = transaction.merge(identity, on="TransactionID", how="left")

# Cast identity categoricals to pandas `category` dtype (NaN preserved).
# DeviceInfo is very high-cardinality -> label-encoded via dtype, NOT one-hot.
identity_cat_cols = [
    c for c in identity.columns
    if c != "TransactionID" and not pd.api.types.is_numeric_dtype(identity[c])
]
for c in identity_cat_cols:
    transaction_id[c] = transaction_id[c].astype("category")

# Re-derive the temporal split on the joined frame (same splitter / val_frac).
train_id_df, val_id_df = temporal_split(transaction_id, val_frac=0.2)
X_train_id, y_train_id = prepare_data(train_id_df)
X_val_id, y_val_id = prepare_data(val_id_df)

print(f"joined frame:        {transaction_id.shape}")
print(f"identity cats cast:  {identity_cat_cols}")
print(f"train (with id):     {X_train_id.shape}   val: {X_val_id.shape}")
print(f"val fraud rate:      {y_val_id.mean():.4f}")

joined frame:        (590540, 248)
identity cats cast:  ['id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
train (with id):     (472432, 246)   val: (118108, 246)
val fraud rate:      0.0344


In [24]:
# BASELINE (tree track, WITH identity) — LightGBM.
# NaN-native; uses the `category` dtype directly (transaction + identity cols).
lgb_id_model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05)
model_lgb_id, auc_lgb_id = train_and_eval(
    lgb_id_model, X_train_id, y_train_id, X_val_id, y_val_id, "baseline_lgbm_identity"
)

2026/08/27 06:00:47 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.146302 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18131
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 246
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784


2026/08/27 06:00:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 06:00:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/27 06:01:

baseline_lgbm_identity: val AUC = 0.8970


In [25]:
# BASELINE (tree track, WITH identity) — XGBoost.
# enable_categorical=True to use the pandas `category` dtype (incl. identity cols).
xgb_id_model = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, enable_categorical=True)
model_xgb_id, auc_xgb_id = train_and_eval(
    xgb_id_model, X_train_id, y_train_id, X_val_id, y_val_id, "baseline_xgb_identity"
)

2026/08/27 06:01:32 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 06:01:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/27 06:01:

baseline_xgb_identity: val AUC = 0.8937


In [26]:
# Quick compare — tree baselines: with vs without identity.
# Pulls the best val_auc per run name from MLflow so it works even after a
# kernel restart (no dependence on earlier in-session variables).
client = MlflowClient()
exp = client.get_experiment_by_name("ieee-fraud-detection")
tree_runs = {}
for r in client.search_runs([exp.experiment_id]):
    name = r.data.tags.get("mlflow.runName", "?")
    if name.startswith(("baseline_lgbm", "baseline_xgb")):
        auc = r.data.metrics.get("val_auc", float("nan"))
        if auc == auc:  # skip NaN
            tree_runs[name] = max(tree_runs.get(name, float("-inf")), auc)

print(f"{'run':<24} {'best_val_auc':>12}")
print("-" * 38)
for name, auc in sorted(tree_runs.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{name:<24} {auc:>12.4f}")

run                      best_val_auc
--------------------------------------
baseline_lgbm                  0.9006
baseline_lgbm_identity         0.8970
baseline_xgb                   0.8960
baseline_xgb_identity          0.8937


## Conclusion & next steps

Adding the raw `identity` table to `transaction` **did not help** — it slightly
*decreased* the validation AUC of both tree models:

| Model | transaction-only | + identity |
|---|---|---|
| LightGBM | 0.9006 | 0.8970 |
| XGBoost | 0.8960 | 0.8937 |

The most likely cause is coverage: `identity` is recorded for only ~25% of
transactions, so its columns are NaN for the remaining ~75%. Combined with the
high-cardinality categoricals (`DeviceInfo`, `id_12`–`id_38`), the raw columns
add noise rather than signal at this stage. The identity data could still help
with **feature engineering** (e.g. a predictive subset, or coverage /
per-device aggregation features) — that can be explored later.

For now I will **fine-tune `baseline_lgbm`**, the best model so far (0.9006).
In production, a simpler model means lower cost and faster inference, so keeping
the transaction-only LightGBM as the base is the pragmatic choice.

## Generate Kaggle submissions

Produce test-set predictions from `baseline_lgbm` and `finetuned_lgbm` so both
can be uploaded to Kaggle and compared.

- **baseline_lgbm** → fixed config `(n_estimators=200, learning_rate=0.05)`.
- **finetuned_lgbm** → best params read back from the MLflow
  `ieee-fraud-detection-finetune` experiment (it lives in `FineTuning.ipynb`, so
  it isn't in this kernel — we rebuild it from the logged config).

Both are (re)trained on `X_train` (the same 80% temporal slice as the validated
models) and predict on the raw test set, keeping exactly the training feature
columns & dtypes. Outputs land in `data/submissions/`.

In [27]:
# Raw test set — keep exactly the training feature columns and dtypes.
from ieee_cis_fraud_detection.config import RAW_DATA_DIR

test = pd.read_csv(RAW_DATA_DIR / "test_transaction.csv")
feature_cols = X_train.columns.tolist()
categorical_cols = [c for c in X_train.columns if X_train[c].dtype.name == "category"]

X_test = test[feature_cols].copy()
for c in categorical_cols:
    X_test[c] = X_test[c].astype("category")

missing = [c for c in feature_cols if c not in test.columns]
assert not missing, f"Test is missing training columns: {missing[:10]}"

print(f"test: {X_test.shape}  (features: {len(feature_cols)}, categorical: {len(categorical_cols)})")

test: (506691, 218)  (features: 218, categorical: 9)


In [28]:
# Rebuild both models and write one Kaggle submission per model.
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(f"sqlite:///{PROJ_ROOT / 'mlruns' / 'mlflow.db'}")
client = MlflowClient()

SUBMISSIONS_DIR = PROJ_ROOT / "data" / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

_INT = {"n_estimators", "num_leaves", "min_child_samples", "subsample_freq", "random_state", "n_jobs", "verbose"}
_FLOAT = {"learning_rate", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda"}


def finetuned_params() -> dict:
    """Best finetuned_lgbm params from MLflow (re-typed from strings)."""
    exp = client.get_experiment_by_name("ieee-fraud-detection-finetune")
    best_run, best_auc = None, float("-inf")
    for r in client.search_runs([exp.experiment_id]):
        auc = r.data.metrics.get("val_auc", float("nan"))
        if auc == auc and auc > best_auc:  # skip failed runs (NaN)
            best_run, best_auc = r, auc
    params = {}
    for k, v in best_run.data.params.items():
        params[k] = int(v) if k in _INT else (float(v) if k in _FLOAT else v)
    print(f"finetuned params (val_auc={best_auc:.4f}): {params}")
    return params


configs = [
    ("baseline_lgbm", {"n_estimators": 200, "learning_rate": 0.05, "n_jobs": -1, "verbose": -1}),
    ("finetuned_lgbm", finetuned_params()),
]

for name, params in configs:
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_test)[:, 1]
    sub = pd.DataFrame({"TransactionID": test["TransactionID"], "isFraud": y_pred})
    out = SUBMISSIONS_DIR / f"{name}_submission.csv"
    sub.to_csv(out, index=False)
    print(f"{name}: wrote {out} ({len(sub):,} rows)")

2026/08/27 12:51:27 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '69b38f3758bf402ab37806596eb9cd9e', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current lightgbm workflow


finetuned params (val_auc=0.9210): {'colsample_bytree': 0.5175897174514872, 'learning_rate': 0.011097554561103107, 'min_child_samples': 49, 'num_leaves': 220, 'reg_alpha': 0.00877781550471966, 'reg_lambda': 0.002273762810253686, 'subsample': 0.9143687545759647, 'n_estimators': 539, 'subsample_freq': 1, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}


2026/08/27 12:52:00 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 12:52:08 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_

baseline_lgbm: wrote /Users/alex/IEEE-CIS_Fraud_Detection_MLOp/data/submissions/baseline_lgbm_submission.csv (506,691 rows)


2026/08/27 12:53:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/27 12:54:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_

finetuned_lgbm: wrote /Users/alex/IEEE-CIS_Fraud_Detection_MLOp/data/submissions/finetuned_lgbm_submission.csv (506,691 rows)
